In [1]:
import pandas as pd, numpy as np, sys, pyarrow, os, re
from unidecode import unidecode
from pathlib import Path

root = Path.cwd().parent/"src"
if str(root) not in sys.path:
    sys.path.append(str(root)) 

pd.set_option('display.max_columns', None)
from routes.Paths import Ruta_opl, Ruta_masterskill, Ruta_arbol_tipificacion, Ruta_global, Ruta_local
df_arbol = pd.read_excel(Ruta_arbol_tipificacion, sheet_name='Tipificación Vista360')
df_arbol = df_arbol[df_arbol.iloc[:,2].notna()].reset_index(drop=True)

df_arbol = df_arbol[df_arbol.iloc[:,2].notna()].reset_index(drop=True)
df_arbol.columns = df_arbol.iloc[1]
df_arbol = df_arbol.iloc[2:].reset_index(drop=True)
df_arbol = df_arbol.rename(columns={
    "PRODUCTO" : "Nivel_1",
    "TIPO" : "Nivel_2",
    "SUBTIPO" : "Nivel_3",
    "¿Debe quedar predefinido?" : "SPC/FCR",
    "Nuevo ANS" : "ANS_Dias",
    "Tipo de día" : "Tipo_Dia",
    "BO A QUE PERTENECE" : "BO_Asignado"
})
df_arbol = df_arbol[
    ["Nivel_1", "Nivel_2", "Nivel_3", "PRI", "SPC/FCR", "ANS_Dias", "Tipo_Dia", "BO_Asignado", "ANS/SLA"]
]

In [2]:
def extraer_sla(texto):

    if pd.isna(texto):
        return pd.Series([0, 'Calendario'])

    texto = str(texto).lower().strip()
    patron = re.search(r'(\d+)\s*d[ií]a[s]?', texto)
    sla_dia = int(patron.group(1)) if patron else 0

    if 'hábil' in texto or 'habil' in texto:
        tipo = 'Habiles'
    elif 'calendario' in texto:
        tipo = 'Calendario'
    else:
        tipo = 'Calendario'

    return pd.Series([sla_dia, tipo])

df_arbol[['SLA_Dia', 'SLA_DiaSLA_Dia']] = df_arbol['ANS/SLA'].apply(extraer_sla)
df_arbol = df_arbol.drop(columns=["ANS/SLA"])

In [3]:
df_arbol = df_arbol.map(lambda x: x.strip() if isinstance(x, str) else x)
df_arbol = df_arbol.map(lambda x: unidecode(x) if isinstance(x, str) else x)
def Columna_fcr(texto_fila):
    if isinstance(texto_fila, str):
        texto_fila = texto_fila.upper().strip()
        if "FCR SI" in texto_fila:
            return "SI"
        elif "FCR NO" in texto_fila:
            return "NO"
    return "N/A"

df_arbol["SPC/FCR"] = df_arbol["SPC/FCR"].apply(Columna_fcr)
df_arbol["ANS_Dias"] = pd.to_numeric(df_arbol["ANS_Dias"], errors="coerce").fillna(0)
df_arbol["BO_Asignado"] = df_arbol["BO_Asignado"].fillna("N/A")
df_arbol["Tipo_Dia"] = df_arbol["Tipo_Dia"].fillna("Calendario")
df_arbol["PRI"] = df_arbol["PRI"].fillna("N/A")
df_arbol = df_arbol.drop_duplicates().reset_index(drop=True)
df_arbol.insert(0, "IdTipificacion", range(1, len(df_arbol) + 1))

cols = ["IdTipificacion", "ANS_Dias"]

df_arbol[cols] = df_arbol[cols].apply(
    pd.to_numeric,
    errors="coerce"
).astype("Int64")

df_arbol.to_csv(os.path.join(Ruta_global, "01.Arbol_tipificaciones.csv"), index=False, encoding="utf-8")
df_arbol.to_csv(os.path.join(Ruta_local, "01.Arbol_tipificaciones.csv"), index=False, encoding="utf-8")

print("Procesamiento exitoso del Arbol de Tipificaciones")

Procesamiento exitoso del Arbol de Tipificaciones
